# LSTM — 30-Trading-Day Return Regression (Colab)

Single-horizon regression model for the active product objective
(`30d-reg-v1`). Predicts `fwd_return_30d = AdjClose[t+30 trading days] / AdjClose[t] - 1`.

**Upload these 3 files** from your local `data/processed/regression_splits_30d/`
(produced by `python src/regression_30d.py export-lstm-splits`):
`train.csv`, `val.csv`, `test.csv`

The features in those CSVs are **already scaled** (the same
winsor -> log1p -> RobustScaler -> clip transform saved in the sklearn
artifact). This notebook does not re-scale features; it only standardizes
the regression target using TRAIN statistics.

Runtime: GPU recommended (Runtime > Change runtime type > GPU).


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
from google.colab import files
print('Select train.csv, val.csv, test.csv (regression_splits_30d) - all 3 at once')
up = files.upload()
assert set(up) >= {'train.csv', 'val.csv', 'test.csv'}, f'missing files, got {list(up)}'


In [ ]:
# Must match src/feature_engineering.py FEATURE_COLS (36) exactly.
FEATURE_COLS = [
    "log_return", "price_to_sma10", "price_to_sma20", "price_to_sma50",
    "price_to_ema20", "price_to_ema50", "price_to_ema200",
    "macd_norm", "macd_hist_norm", "rsi_14", "bb_pct_b", "bb_width",
    "volatility_10", "volatility_20", "roc_5", "roc_10", "roc_20", "roc_40",
    "atr_pct", "volume_ratio", "dist_to_high20", "dist_to_low20",
    "dist_to_high50", "dist_to_low50", "obv_zscore_20",
    "nifty_return", "nifty_volatility_20", "nifty_trend",
    "relative_strength", "sector_return", "vix_zscore_60", "vix_change_5d",
    "fund_roe", "fund_net_profit_growth", "fund_eps_growth", "fund_pe_ratio",
]
TARGET   = "fwd_return_30d"
LOOKBACK = 60
STRIDE   = 5           # 98%+ overlap at stride 1 -> instant train-noise memorisation
N_ENSEMBLE = 5

train = pd.read_csv('train.csv', parse_dates=['Date'])
val   = pd.read_csv('val.csv',   parse_dates=['Date'])
test  = pd.read_csv('test.csv',  parse_dates=['Date'])
print('train', train.shape, 'val', val.shape, 'test', test.shape)
print('feature abs-max (should be <= 5):', float(np.abs(train[FEATURE_COLS].to_numpy()).max()))
print('target mean/std (train):', round(train[TARGET].mean(), 4), round(train[TARGET].std(), 4))


In [ ]:
def make_sequences(df, feature_cols, target, lookback=LOOKBACK, stride=STRIDE):
    """Per-Symbol sliding windows - a window never mixes two companies.
    y is the target on the LAST day of each window ("today")."""
    Xs, ys, meta = [], [], []
    for sym, g in df.groupby('Symbol', sort=False):
        g = g.sort_values('Date').reset_index(drop=True)
        if len(g) < lookback:
            continue
        feats = g[feature_cols].to_numpy(np.float32)
        tgt   = g[target].to_numpy(np.float32)
        w = np.lib.stride_tricks.sliding_window_view(feats, lookback, axis=0).transpose(0, 2, 1)[::stride]
        Xs.append(w)
        ys.append(tgt[lookback - 1::stride])
        meta.append(g.loc[lookback - 1::stride, ['Date', 'Symbol']].reset_index(drop=True))
    return np.concatenate(Xs), np.concatenate(ys), pd.concat(meta, ignore_index=True)

X_train, y_train_raw, m_train = make_sequences(train, FEATURE_COLS, TARGET)
X_val,   y_val_raw,   m_val   = make_sequences(val,   FEATURE_COLS, TARGET)
X_test,  y_test_raw,  m_test  = make_sequences(test,  FEATURE_COLS, TARGET)
print('X_train', X_train.shape, '| X_val', X_val.shape, '| X_test', X_test.shape)

# standardize the target with TRAIN stats only
ty_mean, ty_std = float(y_train_raw.mean()), float(y_train_raw.std())
y_train = (y_train_raw - ty_mean) / ty_std
y_val   = (y_val_raw   - ty_mean) / ty_std
unscale = lambda z: z * ty_std + ty_mean


In [ ]:
def build_lstm(seed):
    tf.random.set_seed(seed)
    l2 = regularizers.l2(1e-4)
    inp = keras.Input(shape=(LOOKBACK, len(FEATURE_COLS)), name='sequence')
    x = layers.LSTM(32, kernel_regularizer=l2, recurrent_dropout=0.2)(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=l2)(x)
    out = layers.Dense(1, name='fwd_return_30d')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4), loss=keras.losses.Huber(), metrics=['mae'])
    return m

build_lstm(0).summary()


In [ ]:
members, histories = [], []
for seed in range(N_ENSEMBLE):
    print(f'=== member {seed + 1}/{N_ENSEMBLE} (seed={seed}) ===')
    m = build_lstm(seed)
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6)]
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=200, batch_size=256, callbacks=cb, verbose=1)
    print(f'  best val_loss = {min(h.history["val_loss"]):.4f}\n')
    members.append(m); histories.append(h)


In [ ]:
# Ensemble prediction on TEST, back in raw-return units.
preds = np.mean([unscale(m.predict(X_test, batch_size=512, verbose=0).ravel()) for m in members], axis=0)
yt = y_test_raw

base = np.full_like(yt, y_train_raw.mean())          # historical-mean baseline
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def mae(a, b):  return float(np.mean(np.abs(a - b)))

report = pd.DataFrame([
    {'model': 'LSTM ensemble',
     'MAE': mae(yt, preds), 'RMSE': rmse(yt, preds),
     'R2': 1 - np.sum((yt - preds) ** 2) / np.sum((yt - yt.mean()) ** 2),
     'MedianAE': float(np.median(np.abs(yt - preds))),
     'DirAcc': float(np.mean(np.sign(yt) == np.sign(preds))),
     'corr': float(np.corrcoef(preds, yt)[0, 1]),
     'pred_mean': float(preds.mean()), 'pred_std': float(preds.std())},
    {'model': 'historical_mean',
     'MAE': mae(yt, base), 'RMSE': rmse(yt, base),
     'R2': 1 - np.sum((yt - base) ** 2) / np.sum((yt - yt.mean()) ** 2),
     'MedianAE': float(np.median(np.abs(yt - base))),
     'DirAcc': float(np.mean(np.sign(yt) == np.sign(base))),
     'corr': 0.0, 'pred_mean': float(base.mean()), 'pred_std': 0.0},
])
print(report.round(5).to_string(index=False))
print()
beat = report.loc[0, 'RMSE'] < report.loc[1, 'RMSE']
print(f'LSTM beats historical-mean baseline on RMSE? {"YES" if beat else "NO"}  '
      f'({(report.loc[0, "RMSE"] / report.loc[1, "RMSE"] - 1) * 100:+.2f}%)')


In [ ]:
plt.figure(figsize=(8, 4))
for i, h in enumerate(histories):
    plt.plot(h.history['loss'], f'C{i}', alpha=.4, label=f'm{i} train')
    plt.plot(h.history['val_loss'], f'C{i}', ls='--', label=f'm{i} val')
plt.xlabel('epoch'); plt.ylabel('Huber loss'); plt.legend(fontsize=7, ncol=2)
plt.title('LSTM training curves (spread = run-to-run instability)'); plt.show()

plt.figure(figsize=(5, 5))
plt.scatter(preds, yt, s=4, alpha=.2)
lim = np.percentile(np.abs(np.r_[preds, yt]), 99)
plt.plot([-lim, lim], [-lim, lim], 'k--', lw=1)
plt.xlabel('predicted 30d return'); plt.ylabel('actual 30d return')
plt.title(f'corr = {np.corrcoef(preds, yt)[0,1]:.3f}'); plt.axis('square'); plt.show()


In [ ]:
# Save the 5 members + target-scaling stats + config, zip, download.
import shutil, os, json as _json
os.makedirs('lstm_reg_30d', exist_ok=True)
for i, m in enumerate(members):
    m.save(f'lstm_reg_30d/member_{i}.keras')
np.savez('lstm_reg_30d/target_scaler.npz', mean=ty_mean, std=ty_std)
_json.dump({'version': 'lstm-30d-reg-v1', 'target': TARGET, 'horizon_trading_days': 30,
            'lookback': LOOKBACK, 'stride': STRIDE, 'n_ensemble': N_ENSEMBLE,
            'feature_cols': FEATURE_COLS, 'features_prescaled': True,
            'target_standardized': True},
           open('lstm_reg_30d/config.json', 'w'), indent=2)
shutil.make_archive('lstm_reg_30d', 'zip', 'lstm_reg_30d')
files.download('lstm_reg_30d.zip')
